In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_DIR = Path.cwd().parent

INPUT_PATH = ROOT_DIR / "data" / "processed" / "hourly_cleaned.parquet"
OUTPUT_PATH = ROOT_DIR / "data" / "processed"

transformed_data = pd.read_parquet(INPUT_PATH)

print("Dataset shape:", transformed_data.shape)
print("Dataset columns:", transformed_data.columns.tolist())

print("Data Types:\n", transformed_data.dtypes)
print("Data Info:")
transformed_data.info()

Dataset shape: (17379, 17)
Dataset columns: ['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']
Data Types:
 instant                int64
dteday        datetime64[us]
season                 int64
yr                     int64
mnth                   int64
hr                     int64
holiday                int64
weekday                int64
workingday             int64
weathersit             int64
temp                 float64
atemp                float64
hum                  float64
windspeed            float64
casual                 int64
registered             int64
cnt                    int64
dtype: object
Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   instant     17379 non-null  int64         
 1   dteday  

In [20]:
# Create timestamp and validate it

transformed_data['timestamp'] = (
    transformed_data['dteday'] + pd.to_timedelta(transformed_data['hr'], unit='h')
)

print("Timestamp range:", transformed_data['timestamp'].min(), "to", transformed_data['timestamp'].max())
print("Timestamp data type:", transformed_data['timestamp'].dtype)
print("Invalid timestamp conversions:", (transformed_data['timestamp'].isna()).sum())

transformed_data[['instant', 'dteday', 'hr', 'timestamp']].head()

Timestamp range: 2011-01-01 00:00:00 to 2012-12-31 23:00:00
Timestamp data type: datetime64[us]
Invalid timestamp conversions: 0


,instant,dteday,hr,timestamp
0,1,2011-01-01,0,2011-01-01 00:00:00
1,2,2011-01-01,1,2011-01-01 01:00:00
2,3,2011-01-01,2,2011-01-01 02:00:00
3,4,2011-01-01,3,2011-01-01 03:00:00
4,5,2011-01-01,4,2011-01-01 04:00:00


In [21]:
# Extracting calender components

transformed_data['year'] = transformed_data['timestamp'].dt.year
transformed_data['month'] = transformed_data['timestamp'].dt.month
transformed_data['day'] = transformed_data['timestamp'].dt.day
transformed_data['hour'] = transformed_data['timestamp'].dt.hour
transformed_data['day_of_week'] = transformed_data['timestamp'].dt.dayofweek
transformed_data['day_name'] = transformed_data['timestamp'].dt.day_name()
transformed_data['quarter'] = transformed_data['timestamp'].dt.quarter

transformed_data['isweekend'] = (
    transformed_data['day_of_week'].isin([5, 6]).astype(int)
)


calendar_features = [
    'timestamp',
    'year',
    'month',
    'day',
    'day_of_week',
    'day_name',
    'quarter',
    'isweekend'
]

transformed_data[calendar_features].sample(15)  #Inspect the new features to ensure they have been created correctly and contain the expected values.

/tmp/ipykernel_11009/1657285974.py:7: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  transformed_data['day_of_week'] = transformed_data['timestamp'].dt.dayofweek


,timestamp,year,month,day,day_of_week,day_name,quarter,isweekend
13049,2012-07-02 22:00:00,2012,7,2,0,Monday,3,0
487,2011-01-22 09:00:00,2011,1,22,5,Saturday,1,1
5477,2011-08-21 03:00:00,2011,8,21,6,Sunday,3,1
3591,2011-06-03 13:00:00,2011,6,3,4,Friday,2,0
9002,2012-01-15 23:00:00,2012,1,15,6,Sunday,1,1
9991,2012-02-26 08:00:00,2012,2,26,6,Sunday,1,1
6417,2011-09-29 23:00:00,2011,9,29,3,Thursday,3,0
14160,2012-08-18 05:00:00,2012,8,18,5,Saturday,3,1
10388,2012-03-13 23:00:00,2012,3,13,1,Tuesday,1,0
2660,2011-04-25 18:00:00,2011,4,25,0,Monday,2,0


In [22]:
consistency_checks = {
    'Year consistency': (
        transformed_data['year'] ==
        (transformed_data['yr'] + 2011)
    ).all(),

    'Month consistency': (
        transformed_data['month'] ==
        transformed_data['mnth']
    ).all(),

    'Weekday consistency': (
        transformed_data['day_of_week'] ==
        transformed_data['weekday']
    ).all(),

    'Weekend consistency': (
        transformed_data['isweekend'] ==
        (transformed_data['weekday'] >= 5).astype(int)
    ).all()
}

for check, result in consistency_checks.items():
    print(f"{check}: {result}")

Year consistency: True
Month consistency: True
Weekday consistency: False
Weekend consistency: False


In [23]:
expected_uci_weekday = (
    transformed_data['day_of_week'] + 1
) % 7

weekday_consistency = (
    transformed_data['weekday'] == expected_uci_weekday
).all()

print("Weekday consistency:", weekday_consistency)

Weekday consistency: True


In [24]:
expected_weekend = (
    transformed_data['day_of_week'].isin([5, 6]).astype(int)
)

weekend_consistency = (
    transformed_data['isweekend'] == expected_weekend
).all()

print("Weekend consistency:", weekend_consistency)

Weekend consistency: True


## 4.3 Categorical Feature Engineering

Convert encoded categorical variables into human-readable labels.

This improves interpretability for exploratory analysis, business reporting,
visualization, and dashboard development.

Original encoded columns will be retained to preserve the source data.

In [25]:
# Converting and mapping encoded categorical features to their original values
# Season labels
season_mapping = {
    1: 'Spring',
    2: 'Summer',
    3: 'Fall',
    4: 'Winter'
}

transformed_data['season_name'] = (
    transformed_data['season'].map(season_mapping)
)

# Weather labels
weather_mapping = {
    1: 'Clear',
    2: 'Mist/Cloudy',
    3: 'Light Rain/Snow',
    4: 'Heavy Rain/Snow'
}

transformed_data['weather_name'] = (
    transformed_data['weathersit'].map(weather_mapping)
)

In [26]:
print("Season labels:")
print(transformed_data['season_name'].value_counts())

print("\nWeather labels:")
print(transformed_data['weather_name'].value_counts())

print("\nMissing season labels:",
      transformed_data['season_name'].isna().sum())

print("Missing weather labels:",
      transformed_data['weather_name'].isna().sum())

Season labels:
season_name
Fall      4496
Summer    4409
Spring    4242
Winter    4232
Name: count, dtype: int64

Weather labels:
weather_name
Clear              11413
Mist/Cloudy         4544
Light Rain/Snow     1419
Heavy Rain/Snow        3
Name: count, dtype: int64

Missing season labels: 0
Missing weather labels: 0


## 4.4 Numerical Feature Engineering

Convert normalized weather measurements into their corresponding
physical units to improve interpretability during business analysis.

Original normalized features will be retained.

In [27]:
# Temperature in Celsius
transformed_data['temp_c'] = transformed_data['temp'] * 39

# Feeling temperature in Celsius
transformed_data['atemp_c'] = transformed_data['atemp'] * 50

# Relative humidity in percentage
transformed_data['humidity_pct'] = transformed_data['hum'] * 100

# Wind speed in km/h
transformed_data['windspeed_kmh'] = transformed_data['windspeed'] * 67

In [28]:
#Validating the new features
weather_features = [
    'temp_c',
    'atemp_c',
    'humidity_pct',
    'windspeed_kmh'
]

print(transformed_data[weather_features].describe())

print("\nMissing values:")
print(transformed_data[weather_features].isna().sum())

             temp_c       atemp_c  humidity_pct  windspeed_kmh
count  17379.000000  17379.000000  17379.000000   17379.000000
mean      19.382500     23.788755     62.722884      12.736540
std        7.509689      8.592511     19.292983       8.196795
min        0.780000      0.000000      0.000000       0.000000
25%       13.260000     16.665000     48.000000       7.001500
50%       19.500000     24.240000     63.000000      12.998000
75%       25.740000     31.060000     78.000000      16.997900
max       39.000000     50.000000    100.000000      56.996900

Missing values:
temp_c           0
atemp_c          0
humidity_pct     0
windspeed_kmh    0
dtype: int64


### Numerical Feature Engineering: Validation Findings

- Converted normalized temperature into Celsius.
- Converted normalized feels-like temperature into Celsius.
- Converted normalized humidity into percentage.
- Converted normalized wind speed into km/h.
- All transformed features contain 17,379 observations.
- No missing values were introduced.
- All converted measurements fall within plausible ranges.

Original normalized columns have been retained for traceability.

## 4.5 Feature Selection and Redundancy Assessment

Identify identifiers, redundant variables, analytical features,
predictive features and target variables.

Preserve the original information until the appropriate
analytical or forecasting stage.

In [29]:
feature_inventory = pd.DataFrame({
    'column': transformed_data.columns,
    'dtype': transformed_data.dtypes.astype(str).values,
    'unique_values': transformed_data.nunique().values,
    'missing_values': transformed_data.isna().sum().values
})

feature_inventory

,column,dtype,unique_values,missing_values
0,instant,int64,17379,0
1,dteday,datetime64[us],731,0
2,season,int64,4,0
3,yr,int64,2,0
4,mnth,int64,12,0
5,hr,int64,24,0
6,holiday,int64,2,0
7,weekday,int64,7,0
8,workingday,int64,2,0
9,weathersit,int64,4,0


In [30]:
# Total rental consistency check
rental_consistency = (
    transformed_data['cnt'] ==
    transformed_data['casual'] + transformed_data['registered']
).all()

print("Rental consistency:", rental_consistency)

Rental consistency: True


In [31]:
# Temporal consistency check
print(
    "Year consistency:",
    (transformed_data['yr'] + 2011 == transformed_data['year']).all()
)

print(
    "Month consistency:",
    (transformed_data['mnth'] == transformed_data['month']).all()
)

print(
    "Hour consistency:",
    (transformed_data['hr'] == transformed_data['hour']).all()
)

print(
    "Temperature conversion:",
    (np.allclose(transformed_data['temp'] * 39,
                 transformed_data['temp_c']))
)

Year consistency: True
Month consistency: True
Hour consistency: True
Temperature conversion: True


In [32]:
# Timestamp uniqueness check
print(
    "Duplicate timestamps:",
    transformed_data['timestamp'].duplicated().sum()
)

print(
    "Chronologically sorted:",
    transformed_data['timestamp'].is_monotonic_increasing
)

Duplicate timestamps: 0
Chronologically sorted: True


### Feature Leakage Considerations

The target variable for total rental demand forecasting is `cnt`.

The variables `casual` and `registered` are components of `cnt`:

cnt = casual + registered

Therefore, they must not be used as independent predictors when
forecasting total demand.

Similarly, record identifiers such as `instant` should not be
treated as meaningful predictive features.

Feature selection will be performed separately for business
analytics and predictive modeling.

In [33]:
# 1. Dataset dimensions
print("Dataset shape:", transformed_data.shape)

# 2. Missing values
print("\nTotal missing values:",
      transformed_data.isna().sum().sum())

# 3. Duplicate records
print("Duplicate records:",
      transformed_data.duplicated().sum())

# 4. Duplicate timestamps
print("Duplicate timestamps:",
      transformed_data['timestamp'].duplicated().sum())

# 5. Chronological order
print("Chronologically sorted:",
      transformed_data['timestamp'].is_monotonic_increasing)

# 6. Rental consistency
print("Rental consistency:",
      (transformed_data['cnt'] ==
       transformed_data['casual'] + transformed_data['registered']).all())

# 7. Datetime validation
print("Timestamp dtype:", transformed_data['timestamp'].dtype)

Dataset shape: (17379, 32)

Total missing values: 0
Duplicate records: 0
Duplicate timestamps: 0
Chronologically sorted: True
Rental consistency: True
Timestamp dtype: datetime64[us]


In [34]:
# Export transformed dataset
csv_output = OUTPUT_PATH / "hourly_transformed.csv"
parquet_output = OUTPUT_PATH / "hourly_transformed.parquet"

transformed_data.to_csv(csv_output, index=False)
transformed_data.to_parquet(parquet_output, index=False)

print("Export completed.")
print("CSV:", csv_output)
print("Parquet:", parquet_output)

Export completed.
CSV: /home/global-rex/Documents/bikeshare-intelligence/data/processed/hourly_transformed.csv
Parquet: /home/global-rex/Documents/bikeshare-intelligence/data/processed/hourly_transformed.parquet
